In [1]:
import cv2
import numpy as np
import mediapipe as mp
import sounddevice as sd
from vosk import Model, KaldiRecognizer
import json
import queue
import threading
import random
import time
import tkinter as tk
from tkinter import ttk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure

In [2]:

# ==============================
# AUDIO 
# ==============================
AUDIO_RATE = 16000
audio_q = queue.Queue()

def audio_callback(indata, frames, time_, status):
    if status:
        print(status)
    audio_data = (indata * 32767).astype(np.int16)
    audio_q.put(audio_data.tobytes())

try:
    audio_stream = sd.InputStream(
        channels=1,
        samplerate=AUDIO_RATE,
        callback=audio_callback
    )
    audio_stream.start()
    print("Audio stream OK.")
except Exception as e:
    print("ERROR con audio:", e)
    exit()

model = Model("model-es/vosk-model-small-es-0.42")
rec = KaldiRecognizer(model, AUDIO_RATE)

# ==============================
# MEDIAPIPE
# ==============================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)
mp_draw = mp.solutions.drawing_utils

# ==============================
# ESTADOS GLOBALES
# ==============================
gesture = None
last_gesture = None
gesture_start_time = 0
gesture_cooldown = 2     # segundos entre acciones de gestos

filter_mode = 0
light_on = True
party_lights = False
confetti_active = False
confetti_intense = False
confetti_intense_end = 0.0

zoom_level = 1.0
voice_command = ""

log_queue = queue.Queue()

# ==============================
# EEG SIMULADO y FUSIÓN
# ==============================
eeg_value = 0.0            # 0..1
eeg_threshold = 0.65       # umbral por defecto
fusion_enabled = True      # activar/desactivar lógica de fusión
fusion_end_time = 0.0      # si fusión temporal, hasta cuándo dura
fusion_duration = 3.0      # segundos de efecto de fusión
intense_overlay = False    # bandera para overlay intenso (activada por fusión)

def log(msg):
    log_queue.put(msg)
    print(msg)

# ==============================
# DETECCIÓN DE GESTOS
# ==============================
def detect_gesture(landmarks):
    y = lambda i: landmarks.landmark[i].y

    if (y(8) < y(6)) and (y(12) < y(10)) and (y(16) > y(14)) and (y(20) > y(18)):
        return "paz"
    if (y(8) > y(6)) and (y(12) > y(10)) and (y(16) > y(14)) and (y(20) > y(18)):
        return "puño"
    if (y(8) < y(6)) and (y(12) < y(10)) and (y(16) < y(14)) and (y(20) < y(18)):
        return "abierta"
    return None

# ==============================
# FILTROS VISUALES (con soporte fusión)
# ==============================
def apply_filters(frame):
    global filter_mode, light_on, party_lights, confetti_active, zoom_level
    global intense_overlay, confetti_intense

    if time.time() < fusion_end_time:
        intense_overlay = True
    else:
        intense_overlay = False

    # Luz (si EEG alto + puño intenso se puede bajar más)
    if not light_on:
        factor = 0.3
        if intense_overlay and last_gesture == "puño" and fusion_enabled:
            factor = 0.12  # mucho más oscuro si la fusión lo manda
        frame = cv2.convertScaleAbs(frame, alpha=factor, beta=0)

    # Zoom
    if zoom_level != 1.0:
        h, w = frame.shape[:2]
        nh, nw = int(h/zoom_level), int(w/zoom_level)
        y1 = (h - nh) // 2
        x1 = (w - nw) // 2
        frame = frame[y1:y1+nh, x1:x1+nw]
        frame = cv2.resize(frame, (w, h))

    # Filtros básicos
    if filter_mode == 1:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    elif filter_mode == 2:
        kernel = np.array([[0.272, 0.534, 0.131],
                           [0.349, 0.686, 0.168],
                           [0.393, 0.769, 0.189]])
        frame = cv2.transform(frame, kernel)
    elif filter_mode == 3:
        edges = cv2.Canny(frame, 100, 200)
        frame = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    elif filter_mode == 10:  # rojo (voz)
        overlay = np.full(frame.shape, (0,0,255), dtype=np.uint8)
        frame = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)
    elif filter_mode == 11:  # azul (voz)
        overlay = np.full(frame.shape, (255,0,0), dtype=np.uint8)
        frame = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)

    # Party lights (si la fusión está activada y EEG alto, hacerlas más intensas)
    if party_lights:
        color = (random.randint(0,255), random.randint(0,255), random.randint(0,255))
        alpha = 0.18
        if intense_overlay and fusion_enabled:
            alpha = 0.45
        overlay = np.full(frame.shape, color, dtype=np.uint8)
        frame = cv2.addWeighted(frame, 1-alpha, overlay, alpha, 0)

    # Confeti normal o intenso
    if confetti_active or (confetti_intense and time.time() < confetti_intense_end):
        count = 20
        if confetti_intense and time.time() < confetti_intense_end:
            count = 120  # muchos más cuando la fusión lo pide
        h,w = frame.shape[:2]
        for _ in range(count):
            x = random.randint(0, w-1)
            y = random.randint(0, h-1)
            cv2.circle(frame, (x,y), random.randint(3,6),
                       (random.randint(0,255), random.randint(0,255), random.randint(0,255)), -1)

    # Si la fusión activa overlay intenso extra (por ejemplo PEACE+EEG)
    if intense_overlay and fusion_enabled:
        overlay = np.full(frame.shape, (random.randint(0,255), random.randint(0,255), random.randint(0,255)), dtype=np.uint8)
        frame = cv2.addWeighted(frame, 0.55, overlay, 0.45, 0)

    # Info rápida en pantalla
    cv2.putText(frame, f"EEG:{eeg_value:.2f} Th:{eeg_threshold:.2f} Fusion:{'ON' if fusion_enabled else 'OFF'}",
                (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    return frame

# ==============================
# EEG SIMULADOR (hilo)
# ==============================
def eeg_simulator():
    global eeg_value
    last_state = False
    while True:
        drift = (0.4 - eeg_value) * 0.03
        
        # ruido normal
        noise = random.uniform(-0.01, 0.01)

        if random.random() < 0.003:
            spike = random.uniform(0.08, 0.10)   
        else:
            spike = 0

        eeg_value = np.clip(eeg_value + drift + noise + spike, 0, 1)

        crossed = eeg_value >= eeg_threshold
        if crossed and not last_state:
            log(f"EEG: {eeg_value:.2f} (>= {eeg_threshold:.2f}) — umbral alcanzado")
        if not crossed and last_state:
            log(f"EEG: {eeg_value:.2f} (< {eeg_threshold:.2f}) — por debajo del umbral")

        last_state = crossed
        time.sleep(0.18)



# ==============================
# FUNCIONES AUX FUSIÓN
# ==============================
def trigger_fusion(source_name):
    """Activa efectos temporales de fusión (overlay intenso)"""
    global fusion_end_time, confetti_intense_end, confetti_intense
    fusion_end_time = time.time() + fusion_duration
    log(f"Fusión: {source_name} + EEG={eeg_value:.2f} → fusión activada por {fusion_duration}s")
    # si el source es confeti, intensificar confeti
    if source_name.lower().startswith("confeti"):
        confetti_intense = True
        confetti_intense_end = time.time() + fusion_duration

# ==============================
# VOZ (HILO) — ahora con fusión
# ==============================
def voice_thread():
    global confetti_active, voice_command, zoom_level, filter_mode

    while True:
        data = audio_q.get()
        if rec.AcceptWaveform(data):
            text = json.loads(rec.Result()).get("text", "")
        else:
            continue

        if not text:
            continue

        voice_command = text
        log(f"Comando de voz: {text}")
        t = text.lower()

        if "rojo" in t:
            filter_mode = 10
            log("Filtro rojo activado")
        if "azul" in t:
            filter_mode = 11
            log("Filtro azul activado")
        if "confeti" in t:
            confetti_active = True
            log("Confeti activado (por voz)")
            # Si fusión habilitada y EEG alto -> confeti intenso
            if fusion_enabled and eeg_value >= eeg_threshold:
                trigger_fusion("Confeti")
        if "acercar" in t:
            zoom_level = min(zoom_level + 0.1, 2.0)
            log("Zoom acercado")
        if "alejar" in t:
            zoom_level = max(zoom_level - 0.1, 1.0)
            log("Zoom alejado")

# ==============================
# TKINTER UI (
# ==============================
def tkinter_ui():
	global filter_mode, light_on, party_lights, confetti_active, zoom_level, eeg_threshold, fusion_enabled

	root = tk.Tk()
	root.title("Panel de Control - Multimodal (EEG + Fusión)")

	# Ayuda
	help_box = tk.LabelFrame(root, text="Ayuda", padx=10, pady=10)
	help_box.grid(row=0, column=0, sticky="nwe", padx=8, pady=6)
	help_text = (
		"GESTOS:\n"
		"✌️ Paz -> Cambia filtro (B/N, sepia, bordes)\n"
		"✊ Puño -> Prender/Apagar luz\n"
		"🖐 Mano abierta -> Luces de fiesta\n\n"
		"COMANDOS DE VOZ:\n"
		"rojo / azul / confeti / acercar / alejar\n\n"
		"FUSIÓN:\n"
		"Si EEG >= umbral y fusión ON, las acciones se intensifican.\n"
	)
	tk.Label(help_box, text=help_text, justify="left").pack()

	# Controles manuales
	btns = tk.LabelFrame(root, text="Controles manuales", padx=10, pady=10)
	btns.grid(row=1, column=0, sticky="we", padx=8, pady=6)

	def btn_reset():
		global filter_mode, light_on, party_lights, confetti_active, zoom_level
		filter_mode = 0
		light_on = True
		party_lights = False
		confetti_active = False
		zoom_level = 1.0
		log("Sistema restablecido manualmente")

	ttk.Button(btns, text="Apagar / Encender Luz", command=lambda: toggle_light()).pack(fill="x", pady=4)
	ttk.Button(btns, text="Cambiar Filtro", command=lambda: cycle_filter()).pack(fill="x", pady=4)
	ttk.Button(btns, text="Reset Total", command=btn_reset).pack(fill="x", pady=4)

	# EEG
	eeg_box = tk.LabelFrame(root, text="EEG (simulado) y fusión", padx=10, pady=10)
	eeg_box.grid(row=2, column=0, sticky="we", padx=8, pady=6)


	fig = Figure(figsize=(4, 1.5), dpi=100)
	ax = fig.add_subplot(111)
	ax.set_ylim(-1, 1)
	ax.set_xlim(0, 200)
	ax.set_title("EEG")

	eeg_line, = ax.plot([], [])
	eeg_data = []

	canvas_eeg = FigureCanvasTkAgg(fig, master=eeg_box)
	canvas_eeg.get_tk_widget().pack(pady=6)

	# === CONTROL DE UMBRAL (igual que antes) ===
	tk.Label(eeg_box, text="Umbral de fusión:").pack()
	thresh_scale = tk.Scale(
		eeg_box, from_=0.0, to=1.0, resolution=0.01, orient="horizontal",
		length=260, command=lambda v: set_threshold(float(v))
	)
	thresh_scale.set(eeg_threshold)
	thresh_scale.pack()

	fusion_var = tk.BooleanVar(value=fusion_enabled)
	def toggle_fusion_ui():
		global fusion_enabled
		fusion_enabled = fusion_var.get()
		log(f"Fusión {'activada' if fusion_enabled else 'desactivada'}")
	tk.Checkbutton(eeg_box, text="Habilitar fusión (EEG + voz/gesto)", variable=fusion_var, command=toggle_fusion_ui).pack(pady=6)

	# Log
	log_box = tk.LabelFrame(root, text="Log", padx=10, pady=10)
	log_box.grid(row=0, column=1, rowspan=3, sticky="nswe", padx=8, pady=6)
	log_text = tk.Text(log_box, height=20, width=60)
	log_text.pack()

	def poll_logs():
		while not log_queue.empty():
			msg = log_queue.get()
			log_text.insert(tk.END, msg + "\n")
			log_text.see(tk.END)

		# actualizar barra EEG
		try:
			eeg_data.append(eeg_value)

			if len(eeg_data) > 200:
				eeg_data.pop(0)

			eeg_line.set_data(range(len(eeg_data)), eeg_data)
			canvas_eeg.draw()

		except Exception:
			pass

		root.after(120, poll_logs)


	poll_logs()
	root.mainloop()

def set_threshold(v):
    global eeg_threshold
    eeg_threshold = float(v)
    log(f"Umbral EEG ajustado a {eeg_threshold:.2f}")

# Botones
def toggle_light():
    global light_on
    light_on = not light_on
    estado = "encendida" if light_on else "apagada"
    log(f"Luz {estado}")

def cycle_filter():
    global filter_mode
    filter_mode = (filter_mode + 1) % 4
    log("Filtro cambiado manualmente")
    log_filter_state()

def log_filter_state():
    if filter_mode == 0:
        log("Filtro: Normal")
    elif filter_mode == 1:
        log("Filtro: B/N")
    elif filter_mode == 2:
        log("Filtro: Sepia")
    elif filter_mode == 3:
        log("Filtro: Bordes")
    elif filter_mode == 10:
        log("Filtro: Rojo (voz)")
    elif filter_mode == 11:
        log("Filtro: Azul (voz)")

# ==============================
# VIDEO THREAD 
# ==============================
def video_thread():
    global last_gesture, gesture_start_time
    global filter_mode, light_on, party_lights, confetti_active

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        log("No se pudo abrir la cámara.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        detected = None

        if result.multi_hand_landmarks:
            for lm in result.multi_hand_landmarks:
                detected = detect_gesture(lm)
                mp_draw.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)

        # GESTO con COOLDOWN
        if detected != last_gesture:
            last_gesture = detected
            gesture_start_time = time.time()

        if detected and time.time() - gesture_start_time >= gesture_cooldown:
            # Cuando se va a ejecutar la acción verificamos EEG + fusion
            is_fused = fusion_enabled and (eeg_value >= eeg_threshold)

            if detected == "paz":
                filter_mode = (filter_mode + 1) % 4
                if is_fused:
                    trigger_fusion("PEACE")
                log(f"Gesto paz ejecutado → cambiar filtro ({'FUSIÓN' if is_fused else 'NORMAL'})")
                log_filter_state()

            if detected == "puño":
                light_on = not light_on
                if is_fused:
                    # si fusion: forzar apagado más fuerte por 3s
                    trigger_fusion("FIST")
                log(f"Gesto puño ejecutado → luz on/off ({'FUSIÓN' if is_fused else 'NORMAL'})")

            if detected == "abierta":
                party_lights = not party_lights
                if is_fused:
                    trigger_fusion("OPEN")
                log(f"Gesto mano abierta ejecutado → luces fiesta ({'FUSIÓN' if is_fused else 'NORMAL'})")

            gesture_start_time = time.time()

        frame = apply_filters(frame)
        cv2.imshow("Multimodal", frame)

        # salir
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

# ==============================
# INICIAR HILOS
# ==============================
threading.Thread(target=eeg_simulator, daemon=True).start()
threading.Thread(target=voice_thread, daemon=True).start()
threading.Thread(target=tkinter_ui, daemon=True).start()
video_thread()


Audio stream OK.


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=11 max-active=4000 lattice-beam=4
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from model-es/vosk-model-small-es-0.42/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from model-es/vosk-model-small-es-0.42/graph/HCLr.fst model-es/vosk-model-small-es-0.42/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading winfo model-es/vosk-model-small-es-0.42/graph/phones/word_boundary.int
I0000 00:00:1765217360.568457   19549 gl_context_egl.cc:85] Successfully ini

Umbral EEG ajustado a 0.65


W0000 00:00:1765217361.273869   20125 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Gesto puño ejecutado → luz on/off (NORMAL)
Gesto puño ejecutado → luz on/off (NORMAL)
Gesto mano abierta ejecutado → luces fiesta (NORMAL)
Comando de voz: fiesta
Gesto mano abierta ejecutado → luces fiesta (NORMAL)
Gesto mano abierta ejecutado → luces fiesta (NORMAL)
Gesto mano abierta ejecutado → luces fiesta (NORMAL)
